# Video Splicer — Dashcam ↔ Proximity CSV Alignment

Splice a dashcam video into `N`-second chunks, where each chunk start corresponds to the row in the proximity CSV whose `timestamp` matches.

For each output splice, the top-left overlay includes:
- `date` (from the CSV row timestamp)
- `start_time` (CSV row timestamp)
- `end_time` (`start_time + N seconds`)

> Assumes the CSV timestamps are aligned to the beginning of the video (we compute video-relative offsets using the **first** CSV timestamp).

## 1) Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import re

VIDEO_PATH = Path("../../Artifacts/input/dashcam_sample.mp4")
CSV_PATH = Path("../../Artifacts/output/proximity_counts_15s.csv")

SPLICE_DURATION_S = 15

MAX_CHUNKS = 5

OUTPUT_BASE_DIR = Path("Artifacts/video_splices")

FONT_SIZE = 24
FONT_COLOR = 'white'
BOX_COLOR = 'black@0.45'
BOX_BORDER_W = 10
TEXT_X = 10
TEXT_Y = 10
LINE_GAP = 32

VIDEO_CODEC = 'libx264'
CRF = 20
PRESET = 'veryfast'
OPENH264_BITRATE = '2M'
MPEG4_Q = 5
VIDEOTOOLBOX_BITRATE = '2M'
AUDIO_CODEC = 'aac'
AUDIO_BITRATE = '96k'

FONTFILE = None



In [ ]:
import pandas as pd
from datetime import timedelta
import subprocess
import math

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"VIDEO_PATH not found: {VIDEO_PATH}")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV_PATH not found: {CSV_PATH}")

if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg not found on PATH. Install ffmpeg and re-run.')

VIDEO_CODEC = 'mpeg4'
print('Using video codec (fixed):', VIDEO_CODEC)

VIDEO_PATH = VIDEO_PATH.resolve()
CSV_PATH = CSV_PATH.resolve()
PROJECT_ROOT = Path('../..').resolve()
if not OUTPUT_BASE_DIR.is_absolute():
    OUTPUT_BASE_DIR = (PROJECT_ROOT / OUTPUT_BASE_DIR).resolve()
else:
    OUTPUT_BASE_DIR = OUTPUT_BASE_DIR.resolve()



In [ ]:
def parse_month_day_folder(stem: str) -> str | None:
    m = re.search(
        r"(january|february|march|april|may|june|july|august|september|october|november|december)[-_ ]?(\d{1,2})",
        stem,
        flags=re.IGNORECASE,
    )
    if not m:
        return None
    month = m.group(1).lower()
    day = int(m.group(2))
    return f"{month}_{day:02d}"

def escape_ffmpeg_text(s: str) -> str:
    s2 = s.replace("\\", "\\\\")
    s2 = s2.replace("'", "\\'")
    s2 = s2.replace(':', '\\:')
    return s2

def datetime_to_video_offset_seconds(ts: pd.Timestamp, video_start_ts: pd.Timestamp) -> float:
    return (ts - video_start_ts).total_seconds()

In [ ]:
def load_and_align_timestamps(csv_path: Path) -> tuple[pd.DataFrame, pd.Timestamp]:
    df = pd.read_csv(csv_path)
    if 'timestamp' not in df.columns:
        raise KeyError("CSV must contain a 'timestamp' column")

    ts = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    if ts.isna().all():
        raise ValueError("Failed to parse any 'timestamp' values as datetimes")

    df = df.loc[~ts.isna()].copy()
    df['timestamp'] = ts.loc[~ts.isna()]

    df = df.sort_values('timestamp').reset_index(drop=True)

    video_start_ts = df.loc[0, 'timestamp']
    return df, video_start_ts

df, video_start_ts = load_and_align_timestamps(CSV_PATH)
print('Rows loaded:', len(df))
print('Baseline (first CSV timestamp):', video_start_ts)


In [ ]:
if len(df) >= 2:
    diffs_s = (df['timestamp'].diff().dt.total_seconds()).dropna()
    if len(diffs_s) > 0:
        median_sampling_s = float(diffs_s.median())
        print('Median CSV step (seconds):', median_sampling_s)
        if not math.isclose(median_sampling_s, float(SPLICE_DURATION_S), rel_tol=0.01, abs_tol=0.25):
            print('WARNING: CSV sampling interval does not closely match SPLICE_DURATION_S.')
else:
    print('Not enough rows to estimate CSV sampling interval.')

rows_to_use = df.iloc[:MAX_CHUNKS] if (MAX_CHUNKS is not None) else df
print('Chunks to generate:', len(rows_to_use))



In [ ]:
folder_candidate = parse_month_day_folder(CSV_PATH.stem) or parse_month_day_folder(VIDEO_PATH.stem)
if folder_candidate is None:
    folder_candidate = video_start_ts.strftime('%Y-%m-%d')

output_dir = OUTPUT_BASE_DIR / folder_candidate
output_dir.mkdir(parents=True, exist_ok=True)
print('Output directory:', output_dir)


In [ ]:
def build_drawtext_filter(date_str: str, start_time_str: str, end_time_str: str) -> str:
    def line(text: str, y: int) -> str:
        esc = escape_ffmpeg_text(text)
        fontfile_opt = f":fontfile={FONTFILE}" if FONTFILE else ''
        return (
            "drawtext="
            f"text='{esc}':"
            f"x={TEXT_X}:y={y}:"
            f"fontsize={FONT_SIZE}:"
            f"fontcolor={FONT_COLOR}:"
            f"box=1:boxcolor={BOX_COLOR}:boxborderw={BOX_BORDER_W}"
            f"{fontfile_opt}"
        )

    y1 = TEXT_Y
    y2 = TEXT_Y + LINE_GAP
    y3 = TEXT_Y + 2 * LINE_GAP

    return ','.join([line(date_str, y1), line(start_time_str, y2), line(end_time_str, y3)])

def splice_one_chunk(row: pd.Series, idx: int) -> dict:
    ts: pd.Timestamp = row['timestamp']

    start_offset_s = datetime_to_video_offset_seconds(ts, video_start_ts)
    end_offset_s = start_offset_s + float(SPLICE_DURATION_S)

    date_str = ts.strftime('%Y-%m-%d')
    start_time_str = ts.strftime('%H:%M:%S')
    end_ts = ts + timedelta(seconds=float(SPLICE_DURATION_S))
    end_time_str = end_ts.strftime('%H:%M:%S')

    ts_compact = ts.strftime('%Y%m%d_%H%M%S')
    out_path = output_dir / f"splice_{idx:06d}_{ts_compact}.mp4"

    vf = build_drawtext_filter(date_str, start_time_str, end_time_str)

    cmd = [
        'ffmpeg',
        '-y',
        '-hide_banner',
        '-loglevel', 'warning',
        '-i', str(VIDEO_PATH),
        '-ss', f"{start_offset_s:.3f}",
        '-t', str(float(SPLICE_DURATION_S)),
        '-vf', vf,
        '-an',
    ]

    if VIDEO_CODEC == 'libx264':
        cmd += ['-c:v', VIDEO_CODEC, '-preset', PRESET, '-crf', str(CRF)]
    elif VIDEO_CODEC == 'libopenh264':
        cmd += ['-c:v', VIDEO_CODEC, '-b:v', OPENH264_BITRATE]
    elif VIDEO_CODEC == 'h264_videotoolbox':
        cmd += ['-c:v', VIDEO_CODEC, '-b:v', VIDEOTOOLBOX_BITRATE]
    else:
        cmd += ['-c:v', VIDEO_CODEC, '-q:v', str(MPEG4_Q)]

    cmd += [str(out_path)]

    print(f"[{idx+1}/{len(rows_to_use)}] {ts.isoformat()} -> {out_path.name}")
    subprocess.run(cmd, check=True)

    return {
        'row_index': int(idx),
        'timestamp': ts.isoformat(),
        'start_offset_s': float(start_offset_s),
        'end_offset_s': float(end_offset_s),
        'date': date_str,
        'start_time': start_time_str,
        'end_time': end_time_str,
        'output_path': str(out_path),
    }


In [ ]:
records: list[dict] = []

for idx, row in rows_to_use.iterrows():
    rec = splice_one_chunk(row, idx=idx)
    records.append(rec)

    if (len(records) % 25) == 0:
        manifest_path = output_dir / 'splice_manifest.csv'
        pd.DataFrame(records).to_csv(manifest_path, index=False)

manifest_path = output_dir / 'splice_manifest.csv'
pd.DataFrame(records).to_csv(manifest_path, index=False)

print('Wrote manifest:', manifest_path)


## Notes

- If overlay placement/font looks off, adjust `FONT_SIZE`, `TEXT_X`, `TEXT_Y`, and `LINE_GAP`.
- Slicing uses each CSV row as the chunk start boundary, and overlays `timestamp` and `timestamp + SPLICE_DURATION_S`.
- Audio is disabled (`-an`) for speed. If you want audio, update the `cmd` in `splice_one_chunk`.
